In [1]:
import pandas as pd
import numpy as np
import os

# 1. LOAD THE 15-YEAR MP BACKBONE
df = pd.read_csv("../data/raw/mp_15yr_climate_backbone.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['District', 'Date'])

# 2. CALCULATE DISTRICT-SPECIFIC THRESHOLDS (Q14 Logic)
# Every district has a different "Normal". We calculate the 95th/5th %ile per district.
print("Computing climatological baselines for 52 districts...")
df['T_95th'] = df.groupby('District')['LST'].transform(lambda x: x.quantile(0.95))
df['SM_5th'] = df.groupby('District')['SSM'].transform(lambda x: x.quantile(0.05))

# 3. CREATE ANOMALY FLAGS
df['Heat_Anomaly'] = (df['LST'] >= df['T_95th']).astype(int)
df['Drought_Anomaly'] = (df['SSM'] <= df['SM_5th']).astype(int)

# 4. INTEGRATE MP CROP PHENOLOGY (Q16 Logic)
df['Month'] = df['Date'].dt.month
# Kharif (Soybean/Cotton) Aug-Sept | Rabi (Wheat) Jan-Feb
df['Is_Kharif_Critical'] = df['Month'].isin([8, 9]).astype(int)
df['Is_Rabi_Critical'] = df['Month'].isin([1, 2]).astype(int)

# 5. DEFINE THE COMPOUND STRESS INDEX (CSI)
# Trigger only if Heat + Drought happen during a critical growth window
df['CSI_Event'] = (df['Heat_Anomaly'] & df['Drought_Anomaly'] & 
                   (df['Is_Kharif_Critical'] | df['Is_Rabi_Critical'])).astype(int)

# 6. GENERATE TARGET (Using the 2018 logic for training)
# If CSI hits, there is an 85% probability of yield failure.
np.random.seed(42)
df['Target_Failure'] = np.where(df['CSI_Event'] == 1, 
                                np.random.choice([1, 0], p=[0.85, 0.15], size=len(df)), 
                                np.random.choice([1, 0], p=[0.02, 0.98], size=len(df)))

# 7. SAVE MASTER TABLE
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/mp_production_master.csv", index=False)
print("✅ MP Production Master Table created with 15 years of daily data!") 

Computing climatological baselines for 52 districts...
✅ MP Production Master Table created with 15 years of daily data!
